In [0]:
from pyspark.sql.functions import col, round as _round

df = spark.table("workspace.dbsf_2305122.sales_raw")
display(df)

order_id,order_date,region,product,quantity,unit_price
1001,2026-01-05,North,Keyboard,3,45.0
1002,2026-01-05,South,Monitor,1,189.5
1003,2026-01-06,East,Keyboard,5,45.0
1004,2026-01-07,North,Mouse,10,17.25
1005,2026-01-08,West,Monitor,2,189.5
1006,2026-01-09,South,Docking Station,4,120.0
1007,2026-01-10,East,Mouse,7,17.25
1008,2026-01-11,North,Monitor,3,189.5
1009,2026-01-12,West,Keyboard,2,45.0
1010,2026-01-12,South,Mouse,15,17.25


In [0]:
df.select("order_id", "region", "quantity").show()

+--------+------+--------+
|order_id|region|quantity|
+--------+------+--------+
|    1001| North|       3|
|    1002| South|       1|
|    1003|  East|       5|
|    1004| North|      10|
|    1005|  West|       2|
|    1006| South|       4|
|    1007|  East|       7|
|    1008| North|       3|
|    1009|  West|       2|
|    1010| South|      15|
|    1011|  East|       1|
|    1012| North|       2|
+--------+------+--------+



In [0]:
df.select(
    col("order_id").alias("id"),
    col("unit_price").alias("price")
).show(5)

+----+-----+
|  id|price|
+----+-----+
|1001| 45.0|
|1002|189.5|
|1003| 45.0|
|1004|17.25|
|1005|189.5|
+----+-----+
only showing top 5 rows


In [0]:
%sql
SELECT order_id AS id, unit_price AS price
FROM workspace.dbsf_2305122.sales_raw
LIMIT 5;

id,price
1001,45.0
1002,189.5
1003,45.0
1004,17.25
1005,189.5


I found SQL easier to read and I would prefer to maintain SQL in a file of 200 lines because it is concise and easier to understand.

In [0]:
busy = df.filter(col("quantity") >= 5)
display(busy)
print(busy.count(), "rows")

order_id,order_date,region,product,quantity,unit_price
1003,2026-01-06,East,Keyboard,5,45.0
1004,2026-01-07,North,Mouse,10,17.25
1007,2026-01-10,East,Mouse,7,17.25
1010,2026-01-12,South,Mouse,15,17.25


4 rows


In [0]:
df.filter((col("region") == "North") &
         (col("product") == "Keyboard")).show()

+--------+----------+------+--------+--------+----------+
|order_id|order_date|region| product|quantity|unit_price|
+--------+----------+------+--------+--------+----------+
|    1001|2026-01-05| North|Keyboard|       3|      45.0|
+--------+----------+------+--------+--------+----------+



In [0]:
df.filter((col("region") == "West") |
         (col("quantity") > 10)).show()

+--------+----------+------+--------+--------+----------+
|order_id|order_date|region| product|quantity|unit_price|
+--------+----------+------+--------+--------+----------+
|    1005|2026-01-08|  West| Monitor|       2|     189.5|
|    1009|2026-01-12|  West|Keyboard|       2|      45.0|
|    1010|2026-01-12| South|   Mouse|      15|     17.25|
+--------+----------+------+--------+--------+----------+



In [0]:
df.filter(col("region") == "North" and
          col("product") == "Keyboard").show()

---------------------------------------------------------------------------
PySparkValueError                         Traceback (most recent call last)
File <command-4873815451999001>, line 1
----> 1 df.filter(col("region") == "North" and
      2           col("product") == "Keyboard").show()

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/column.py:635, in Column.__nonzero__(self)
    634 def __nonzero__(self) -> None:
--> 635     raise PySparkValueError(
    636         errorClass="CANNOT_CONVERT_COLUMN_INTO_BOOL",
    637         messageParameters={},
    638     )

PySparkValueError: [CANNOT_CONVERT_COLUMN_INTO_BOOL] Cannot convert column into bool: please use '&' for 'and', '|' for 'or', '~' for 'not' when building DataFrame boolean expressions.

The Python keyword and does not work because a Spark Column represents a condition for multiple rows rather than one Boolean value; Spark uses & for AND between column conditions.

In [0]:
priced = df.withColumn("revenue",
    col("quantity") * col("unit_price"))

display(priced)

order_id,order_date,region,product,quantity,unit_price,revenue
1001,2026-01-05,North,Keyboard,3,45.0,135.0
1002,2026-01-05,South,Monitor,1,189.5,189.5
1003,2026-01-06,East,Keyboard,5,45.0,225.0
1004,2026-01-07,North,Mouse,10,17.25,172.5
1005,2026-01-08,West,Monitor,2,189.5,379.0
1006,2026-01-09,South,Docking Station,4,120.0,480.0
1007,2026-01-10,East,Mouse,7,17.25,120.75
1008,2026-01-11,North,Monitor,3,189.5,568.5
1009,2026-01-12,West,Keyboard,2,45.0,90.0
1010,2026-01-12,South,Mouse,15,17.25,258.75


In [0]:
priced = priced.withColumn("revenue_rounded",
    _round(col("revenue"), 0))

priced.select("order_id", "revenue", "revenue_rounded").show()

+--------+-------+---------------+
|order_id|revenue|revenue_rounded|
+--------+-------+---------------+
|    1001|  135.0|          135.0|
|    1002|  189.5|          190.0|
|    1003|  225.0|          225.0|
|    1004|  172.5|          173.0|
|    1005|  379.0|          379.0|
|    1006|  480.0|          480.0|
|    1007| 120.75|          121.0|
|    1008|  568.5|          569.0|
|    1009|   90.0|           90.0|
|    1010| 258.75|          259.0|
|    1011|  120.0|          120.0|
|    1012|  240.0|          240.0|
+--------+-------+---------------+



In [0]:
df.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- order_date: date (nullable = true)
 |-- region: string (nullable = true)
 |-- product: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: double (nullable = true)



DataFrames are immutable, so withColumn creates a new DataFrame instead of modifying the original DataFrame.

In [0]:
chain = (df
    .filter(col("quantity") >= 5)
    .withColumn("revenue", col("quantity") * col("unit_price"))
    .select("order_id", "region", "revenue"))

chain.explain()

== Physical Plan ==
PhotonResultStage
+- PhotonColumnarToRow
   +- PhotonProject [order_id#12619, region#12621, (cast(quantity#12623 as double) * unit_price#12624) AS revenue#12628]
      +- PhotonScan parquet workspace.dbsf_2305122.sales_raw[order_id#12619,region#12621,quantity#12623,unit_price#12624] DataFilters: [isnotnull(quantity#12623), (quantity#12623 >= 5)], DictionaryFilters: [(quantity#12623 >= 5)], Format: parquet, Location: PreparedDeltaFileIndex(1 paths)[s3://dbstorage-prod-gqmz0tfnwy/uc/8d6d1905-2456-400d-846f-3145e9b..., OptionalDataFilters: [], PartitionFilters: [], ReadSchema: struct<order_id:int,region:string,quantity:int,unit_price:double>, RequiredDataFilters: [isnotnull(quantity#12623), (quantity#12623 >= 5)]


== Photon Explanation ==
The query is fully supported by Photon.
== Optimizer Statistics (table names per statistics state) ==
  missing = 
  partial = 
  full    = sales_raw



No Exchange operation appears in the physical plan, which shows that the transformations are narrow and no shuffle occurred.